In [ ]:
import sys
import gc
from pathlib import Path
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader 

import torch
from neuralop.models import FNO
from neuralop import Trainer
from neuralop.training import AdamW
from neuralop import LpLoss, H1Loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torch')
warnings.filterwarnings('ignore', category=UserWarning, module='neuralop')

sys.path.insert(0, '../loc_libs')
from read_3d_root import read_3d_data
from read_3d_hdf5 import read_3d_data_hdf5

from norm_utils import build_normalizer
from fluid_datasets import FluidDataset
from fluid_datasets import FluidDatasetH5

In [ ]:
FEATURE_NAMES = ['ed', 'vx', 'vy', 'vz']

lazy = False
full_paths = ['../data/dAu_50ev_mb_test.h5']
nevents_in = -1
start_event = 0


In [ ]:
print(f'\nLoading {len(full_paths)} HDF5 file(s)  lazy={lazy} ...')

dat = read_3d_data_hdf5(
    full_paths,
    nevents=nevents_in,
    start_event=start_event,
    lazy=lazy,
    verbose=True,
)

In [ ]:
for k, v in dat.items():
    if k not in ('arr', 'tau_freezeout', 'ntau_freezeout', 'nans'):
        print(f'  {k}: {v}')

arr         = dat['arr']
grid_params = {k: dat[k] for k in ('tau_min', 'dtau', 'eta_min', 'deta') if k in dat}

In [ ]:
n_total = len(arr)
n_train = int(n_total * 0.8)
print(f'total events: {n_total}  →  train={n_train}  val={n_total - n_train}')

norm_name = {'name': 'none'}

normalizer = build_normalizer(norm_name,grid_params)
print(f'normalizer: {normalizer}')

norm_name2 = {'name': 'etau75_3d'}

normalizer_etau75_3d = build_normalizer(norm_name2,grid_params)
print(f'normalizer_etau75_3: {normalizer_etau75_3d}')

normalizer_etau75_eta_shape_3d = build_normalizer({'name': 'etau75_eta_shape_3d', 'shape_ref': (n_train, *arr.shape[1:])}, grid_params)
print(f'normalizer_etau75_eta_shape_3d: {normalizer_etau75_eta_shape_3d}')


In [ ]:
xyslice     = slice(None, -1)   # drop last x/y point (matches train_model.py)
tauslice    = slice(None, -1)   # drop last tau point  (matches train_model.py)

arr = arr[:, :, xyslice, xyslice, tauslice, :]

print(f'shapes train:  {n_train} events  event_shape={arr.shape}')


In [ ]:
def plot_mean_max_energy_vs_eta(arr, n_events=None, feature_idx=0, tau_bin=0):
    """
    Plot the mean of the per-event maximum feature value as a function of eta bin
    at one or more fixed tau bins, overlaid on the same figure.

    For each event and each eta bin the maximum is taken over all (x, y)
    positions at the given tau bin.  Those per-event maxima are then averaged
    over the requested number of events.

    Parameters
    ----------
    arr         : np.ndarray, shape (n_events, n_features, nx, ny, n_eta, n_tau)
    n_events    : int or None — number of events to include (None = all)
    feature_idx : int — channel index (0=ed, 1=vx, 2=vy, 3=vz)
    tau_bin     : int or list of int — time step index/indices along the tau axis
    """
    total_events, n_features, nx, ny, n_eta, n_tau = arr.shape

    if n_events is None or n_events > total_events:
        n_events = total_events

    feature_idx = int(np.clip(feature_idx, 0, n_features - 1))
    fname = FEATURE_NAMES[feature_idx] if feature_idx < len(FEATURE_NAMES) else str(feature_idx)

    tau_bins = [tau_bin] if isinstance(tau_bin, (int, np.integer)) else list(tau_bin)
    tau_bins = [int(np.clip(t, 0, n_tau - 1)) for t in tau_bins]

    eta_bins = np.arange(n_eta)

    fig, ax = plt.subplots(figsize=(8, 4))

    for tb in tau_bins:
        # select tau bin; shape → (n_events, nx, ny, n_eta)
        data = arr[:n_events, feature_idx, :, :, :, tb]

        # max over (x, y) for each (event, eta_bin) → shape (n_events, n_eta)
        max_per_event = data.reshape(n_events, nx * ny, n_eta).max(axis=1)

        # mean over events → shape (n_eta,)
        mean_max = max_per_event.mean(axis=0)
        std_max  = max_per_event.std(axis=0)

        line, = ax.plot(eta_bins, mean_max, marker='o', linewidth=1.5, label=f'tau_bin={tb}')
        ax.fill_between(eta_bins, mean_max - std_max, mean_max + std_max,
                        alpha=0.25, color=line.get_color())

    ax.set_xlabel('eta bin index')
    ax.set_ylabel(f'mean max {fname}')
    tau_label = str(tau_bins[0]) if len(tau_bins) == 1 else str(tau_bins)
    ax.set_title(f'Mean of max {fname} vs. eta bin  ({n_events} events, tau_bin={tau_label})')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
#plot_mean_max_energy_vs_eta(arr[0:n_train], n_events=n_train, feature_idx=0, tau_bin=0)
m_tau_bins = [0, 3, 5, 10, 20]
plot_mean_max_energy_vs_eta(arr[0:n_train], n_events=n_train, feature_idx=0, tau_bin=m_tau_bins)

In [ ]:
#plot_mean_max_energy_vs_eta(normalizer_etau75_3d.fit_transform(arr[0:n_train]) , n_events=n_train, feature_idx=0, tau_bin=20)
#arr_norm2 = normalizer_etau75_3d.fit_transform(arr[0:n_train])
arr_norm2 = normalizer_etau75_eta_shape_3d.fit_transform(arr[0:n_train])
plot_mean_max_energy_vs_eta(arr_norm2, n_events=n_train, feature_idx=0, tau_bin=m_tau_bins)

In [ ]:
#plot_mean_max_energy_vs_eta(normalizer_etau75_3d.inverse(arr_norm2), n_events=n_train, feature_idx=0, tau_bin=m_tau_bins)
plot_mean_max_energy_vs_eta(normalizer_etau75_eta_shape_3d.inverse(arr_norm2), n_events=n_train, feature_idx=0, tau_bin=m_tau_bins)

In [ ]:
#plt.imshow(normalizer_etau75_eta_shape_3d.ref)
normalizer_etau75_eta_shape_3d.ref.shape

In [ ]:
def plot_eta_bins(arr, event_idx=0, tau_bin=0, feature_idx=0):
    """
    Display x-y heatmaps for every eta bin at a fixed tau bin and feature.

    Parameters
    ----------
    arr        : np.ndarray, shape (n_events, n_features, nx, ny, n_eta, n_tau)
    event_idx  : which event to display
    tau_bin    : time step index along the tau (last) axis
    feature_idx: channel index (0=ed, 1=vx, 2=vy, 3=vz)
    """
    n_events, n_features, nx, ny, n_eta, n_tau = arr.shape

    # ── bounds checking ──────────────────────────────────────────────────────
    event_idx   = int(np.clip(event_idx,   0, n_events   - 1))
    tau_bin     = int(np.clip(tau_bin,     0, n_tau      - 1))
    feature_idx = int(np.clip(feature_idx, 0, n_features - 1))

    fname = FEATURE_NAMES[feature_idx] if feature_idx < len(FEATURE_NAMES) else str(feature_idx)

    # select the 2D slices for all eta bins: shape (n_eta, nx, ny)
    # arr[event, feature, x, y, eta, tau]
    #slices = arr[event_idx, feature_idx, :, :, :, tau_bin]   # (nx, ny, n_eta)
    slices = arr[event_idx, feature_idx, :, :, :, tau_bin]
    denom = arr[event_idx, feature_idx, :, :, 8, tau_bin]
    #slices = slices - denom[..., np.newaxis]
    slices = np.moveaxis(slices, -1, 0)                       # (n_eta, nx, ny)

    # shared colour scale across all eta bins for fair comparison
    vmin, vmax = slices.min(), slices.max()

    # ── layout ───────────────────────────────────────────────────────────────
    ncols = min(n_eta, 4)
    nrows = int(np.ceil(n_eta / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 3.2, nrows * 3.0),
        squeeze=False,
    )
    fig.suptitle(
        f'feature={fname}  |  event={event_idx}  |  tau_bin={tau_bin}',
        fontsize=13, y=1.01,
    )

    for i_eta in range(n_eta):
        row, col = divmod(i_eta, ncols)
        ax = axes[row][col]
        im = ax.imshow(
            slices[i_eta].T,          # transpose so x→horizontal, y→vertical
            origin='lower',
            aspect='equal',
            vmin=vmin, vmax=vmax,
            cmap='viridis',
        )
        ax.set_title(f'eta bin {i_eta}', fontsize=9)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # hide any unused subplots
    for i_empty in range(n_eta, nrows * ncols):
        row, col = divmod(i_empty, ncols)
        axes[row][col].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
#plot_eta_bins(arr, event_idx=0, tau_bin=1, feature_idx=0)

In [ ]:
#print('[norm] data-driven normalizer: materialising training split for ref ...')
#normalizer.fit_transform(arr[0:n_train])   # populates normalizer.ref; result discarded
#gc.collect()
#print(f'[norm] ref computed  shape={normalizer.ref.shape}')

In [ ]:
#plot_eta_bins(arr, event_idx=0, tau_bin=1, feature_idx=0)